# LES–FSI analysis: flexible rectangular membrane

Step-by-step analysis notebook.

**Physics**
- Fluid: Smagorinsky LES, finite-volume
- Solid: flexible cantilever membrane (inertia + bending + damping)
- Interface: moving no-slip on the membrane

**Note:** the sphere package in `../working/` is unchanged. Run this notebook from `membrane_fsi/`.

## Step 1 — Setup working directory and imports

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display, Markdown

# Make sure we are inside membrane_fsi/
cwd = Path.cwd().resolve()
if (cwd / "les_membrane_fsi.py").exists():
    root = cwd
elif (cwd.parent / "les_membrane_fsi.py").exists():
    root = cwd.parent
    os.chdir(root)
else:
    raise FileNotFoundError(
        "Cannot find les_membrane_fsi.py. Start Jupyter from the membrane_fsi folder."
    )

sys.path.insert(0, str(root))
print("Working directory:", root)
print("Files:", sorted([p.name for p in root.iterdir()])[:20])

## Step 2 — Install check (optional)

If imports fail, run this cell once.

In [ ]:
# %pip install -r requirements.txt

## Step 3 — Build / load the membrane mesh

Creates (if missing):
- `data/fluid_mesh_membrane.msh`
- `data/processed_mesh_membrane.npz`

In [ ]:
from run import ensure_mesh

npz_path, geom_meta = ensure_mesh(coarse=True)
print("Processed mesh:", npz_path)
print("Geometry meta:", geom_meta)

mesh = np.load(npz_path)
print("Mesh keys:", sorted(mesh.files))
print("Cells:", len(mesh["cell_volumes"]))
print("Membrane faces:", int(np.sum(mesh["boundary_tag"] == 4)))
print("Inlet/outlet/walls:",
      int(np.sum(mesh["boundary_tag"] == 1)),
      int(np.sum(mesh["boundary_tag"] == 2)),
      int(np.sum(mesh["boundary_tag"] == 3)))

## Step 4 — Configure and run the LES–FSI simulation

- `quick = True` → short demo
- `quick = False` → longer analysis run

In [ ]:
from les_membrane_fsi import MembraneFSISolver, FluidConfig

quick = True  # set False for a longer run

if quick:
    cfg = FluidConfig(dt=1e-4, t_end=0.02, print_every=40, frame_every=20, inlet_ti=0.01)
    max_steps = 120
else:
    cfg = FluidConfig(dt=1e-4, t_end=0.08, print_every=50, frame_every=25, inlet_ti=0.01)
    max_steps = 500

solver = MembraneFSISolver(str(npz_path), fluid_cfg=cfg, geom_meta=str(geom_meta))
solver.run(max_steps=max_steps, capture_gif=True)

print("Final tip deflection η =", solver.solid.tip_deflection())
print("Final tip velocity     =", float(solver.solid.v[-1]))

## Step 5 — Save analysis outputs

In [ ]:
out = Path("outputs")
out.mkdir(exist_ok=True)

fields = out / "les_membrane_fields.npz"
solver.save_fields(str(fields))

outputs = {
    "midplane": solver.plot_midplane(str(out / "flow_past_membrane_midplane.png")),
    "deflection": solver.plot_membrane_response(str(out / "membrane_deflection.png")),
    "gif": solver.save_gif(str(out / "flow_past_membrane.gif")),
}
outputs

## Step 6 — Visual analysis (plots + GIF)

In [ ]:
display(Markdown("### Animation (flow + membrane tip motion indicator)"))
display(Image(filename=outputs["gif"]))

display(Markdown("### Mid-plane velocity field"))
display(Image(filename=outputs["midplane"]))

display(Markdown("### Membrane structural response"))
display(Image(filename=outputs["deflection"]))

## Step 7 — Quantitative analysis

Inspect tip history, deflection shape, and basic fluid statistics.

In [ ]:
data = np.load(fields)
print("Saved field keys:", sorted(data.files))
print("Tip deflection:", float(data["tip_eta"]))
print("eta range:", float(data["membrane_eta"].min()), "→", float(data["membrane_eta"].max()))
print("membrane velocity range:", float(data["membrane_v"].min()), "→", float(data["membrane_v"].max()))

Umag = np.linalg.norm(data["U"], axis=1)
print("|U| mean/max:", float(Umag.mean()), float(Umag.max()))
print("pressure range:", float(data["p"].min()), float(data["p"].max()))
print("nu_t mean:", float(data["nu_t"].mean()))

In [ ]:
# Tip deflection history from solver.history
if solver.history:
    t = np.array([h["t"] for h in solver.history])
    tip = np.array([h["tip_eta"] for h in solver.history])

    fig, ax = plt.subplots(figsize=(7, 3.5))
    ax.plot(t, tip, "-o", ms=3)
    ax.set_xlabel("t")
    ax.set_ylabel("tip deflection η")
    ax.set_title("Membrane tip response vs time")
    ax.grid(True, alpha=0.3)
    plt.show()
else:
    print("No history recorded.")

In [ ]:
# Current deflection shape along membrane height
fig, ax = plt.subplots(figsize=(5.5, 4))
ax.plot(data["membrane_eta"], data["membrane_z"], "o-", color="crimson")
ax.axvline(0.0, color="k", ls=":", lw=1)
ax.set_xlabel("streamwise deflection η")
ax.set_ylabel("z (height)")
ax.set_title("Membrane shape (clamped at bottom)")
ax.grid(True, alpha=0.3)
plt.show()

## Step 8 — What to conclude

Check these points in your results:
1. Flow approaches from the inlet and interacts with the plate.
2. Membrane tip deflection/velocity becomes nonzero (flexible solid response).
3. No-slip is moving: wall velocity follows membrane motion.
4. LES eddy viscosity `nu_t` is nonzero near shear regions.

All analysis files are in `membrane_fsi/outputs/`.